# Structured execution

A statevector simulator pays one pass over all $2^n$ amplitudes per gate.
Many circuits are built from pieces whose *meaning* is much cheaper than
their gates: an adder or a streaming step only moves amplitudes between
basis states, a small block acts on a handful of qubits, and phase
estimation applies the same controlled unitary over and over.

`Block.statevector` and `QarpEngine` read the block tree before simulating
and run each piece as the cheapest exact kernel — a permutation as one
gather, a small block as one dense matrix, a controlled-power ladder as one
kernel — and everything else as its (fused) gates.  Results are exact,
global phase included; a circuit with no such structure runs exactly as
before.

## A permutation, one gather

An increment $x \to x + 1 \bmod 2^k$ written as a ladder of multi-controlled
flips.  Repeated on two interleaved registers of a 16-qubit state it is
hundreds of gates, but as a whole it only permutes basis states.

In [ ]:
import time

import numpy as np

from qarp.blocks import CompositeBlock, SimpleBlock


class Increment(SimpleBlock):
    def build_vanilla(self):
        for t in reversed(range(1, self.n_qubits)):
            self.mcx(*range(t), t)
        self.x(0)


n = 16
step = CompositeBlock(
    [Increment(8, target_qubits=list(range(0, n, 2))), Increment(8, target_qubits=list(range(1, n, 2)))],
    n_qubits=n,
)
block = (step ** 8).build()

rng = np.random.default_rng(0)
psi = rng.standard_normal(1 << n) + 1j * rng.standard_normal(1 << n)
psi /= np.linalg.norm(psi)

t = time.perf_counter()
gates = block.statevector(psi, structured=False)
t_gates = time.perf_counter() - t
block.statevector(psi, structured=True)  # plans once; the plan is cached on the block
t = time.perf_counter()
structured = block.statevector(psi, structured=True)
t_structured = time.perf_counter() - t

print(f"{len(block.flatten())} gates: {t_gates * 1e3:.1f} ms as gates, {t_structured * 1e3:.2f} ms structured")
print("max |difference|:", np.max(np.abs(gates - structured)))

## Declaring the action

A block that knows it is a permutation can say so with `classical_action`:
it maps local basis indices to their images (int64, LSB first).  The
declaration is a promise — $U|x\rangle = |f(x)\rangle$ exactly, phase
included — and `ModularMultiplicationBlock` makes it for $x \to m x \bmod N$.

In [ ]:
from qarp.blocks import ModularMultiplicationBlock

mul = ModularMultiplicationBlock(7, 15)
print(mul.classical_action(np.arange(16)))

## Phase estimation: a declared ladder, never built

With eleven ancillas, QPE's controlled-$U$ ladder is $2^{11} - 1$ applications
of the same block.  `QPEBlock` declares that with `structure()`: its parts in
order, where each ancilla's applications are one `Repeat`.  The engine reads
the declaration before the block is built and runs the ladder as $U^{2^j}$
under ancilla $j$ from one matrix, so the phase $3/8$ of $P(2\pi \cdot 3/8)$
reads out exactly, the ladder is never built, and `qpe.block` stays `None`.

In [ ]:
import qarp
from qarp.algorithms import QPE
from qarp.blocks import ComputationalBasisStateBlock, QPEBlock, Repeat
from qarp.engines import QarpEngine

u = SimpleBlock(1)
u.p(0, 2 * np.pi * 0.375)
u.build()

parts = QPEBlock(ComputationalBasisStateBlock([1]), u, 11, 1).structure()
print([f"{p.block.name} x {p.count}" if isinstance(p, Repeat) else p.name for p in parts])

t = time.perf_counter()
qpe = QPE(
    state=ComputationalBasisStateBlock([1]),
    unitary=u,
    n_ancilla=11,
    engine=QarpEngine(n_shots=qarp.EXACT),
).build()
print(f"built in {(time.perf_counter() - t) * 1e3:.1f} ms; block built: {qpe.block is not None}")
print("phase:", qpe.run(), " probability:", qpe.result_probability)

## The knobs

`structured=False` on `Block.statevector` or `QarpEngine` runs the gate
stream as-is, and makes `QPE` build its full block; `QARP_STRUCTURED=0` does
the same process-wide.  Engines with a `device`, amplitude-consuming
primitives and gradients always run the gate path.

`optimization_level` (0, 1 or 2, as in `Block.optimize`) runs the transpiler
after planning, on the gate slices only, or on the whole stream on the gate
path.  `Block.statevector` and `Block.kernels` take it per call, and `None`
runs the slices as they are; `QarpEngine` takes it once and defaults to 1.
`Block.optimize` flattens the tree first, which leaves the planner nothing to
read: below, the permutation survives `optimization_level=1` but not
`optimize(level=1)`.

In [ ]:
layer = SimpleBlock(n)
for q in range(n):
    layer.rz(q, 0.3)
    layer.rz(q, 0.4)
    layer.h(q)
    layer.h(q)
for q in range(n - 1):
    layer.cx(q, q + 1)
mixed = CompositeBlock([Increment(8, target_qubits=list(range(0, n, 2))), layer], n_qubits=n)
mixed.build()

print("kernels at level 1:", mixed.kernels(optimization_level=1))
print(f"gate slice: {len(layer.flatten())} gates, {len(layer.optimize(level=1).flatten())} at level 1")
print("kernels after optimize(level=1):", mixed.optimize(level=1).kernels())

reference = mixed.statevector(psi, structured=False)
for level in (None, 0, 1, 2):
    difference = np.max(np.abs(mixed.statevector(psi, optimization_level=level) - reference))
    print(f"level {level}: max |difference| {difference:.1e}")